# Colab Training Notebook — EVM-nano (VOC → COCO → Export → Bench)

Full pipeline, same stages as the platform-agnostic twin `Generic_Notebook.ipynb`:

**overfit gate → VOC full → full COCO → ONNX/INT8 export → latency bench → curves**

- Runtime: **GPU (T4)** — Runtime → Change runtime type
- Artifacts sync to Drive (`MyDrive/Edge_Vision_Model/`); training auto-resumes from Drive after disconnects
- Datasets download from official mirrors at runtime (never stored in the repo)
- Overfit gate (20 images, mAP@0.5 ≥ 0.90) **must pass** before any full training


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Edge_Vision_Model'
import os
os.makedirs(DRIVE, exist_ok=True)
print('Drive artifacts root:', DRIVE)


In [ ]:
!nvidia-smi -L || echo 'no GPU - switch runtime type'
!python -V


## 1. Setup — clone repo + deps


In [ ]:
%cd /content
!rm -rf edge-vision-model
!git clone https://github.com/avneeshjadhav04/edge-vision-model
%cd edge-vision-model
!pip -q install pyyaml onnx onnxruntime onnxsim opencv-python-headless
import sys
sys.path.insert(0, '/content/edge-vision-model')
DATA = '/content/datasets'
RUNS = '/content/runs'
import os
os.makedirs(DATA, exist_ok=True)
os.makedirs(RUNS, exist_ok=True)
print('data:', DATA, '| runs:', RUNS)


## 2. VOC data (official mirrors, runtime download)


In [ ]:
from data.download import download_voc
import glob
download_voc(root=f'{DATA}/VOC')
for f in glob.glob(f'{DATA}/VOC/*.tar'):
    os.remove(f)
print('VOC ready at', f'{DATA}/VOC')


## 3. Overfit sanity gate — 20 images, mAP@0.5 ≥ 0.90 or STOP

Fast correctness check (~5–10 min on T4). If this fails, do **not** start full training.


In [ ]:
import subprocess, sys
cmd = [sys.executable, '-m', 'scripts.overfit_test', '--root', f'{DATA}/VOC',
       '--epochs', '300', '--batch-size', '8', '--img-size', '320',
       '--device', 'cuda', '--save-dir', f'{RUNS}/overfit']
print(' '.join(cmd))
rc = subprocess.run(cmd).returncode
os.makedirs(f'{RUNS}/overfit', exist_ok=True)
gate = 'PASS' if rc == 0 else 'FAIL'
with open(f'{RUNS}/overfit/GATE', 'w') as f:
    f.write(gate)
print('GATE:', gate)
assert gate == 'PASS', 'Overfit gate FAILED — fix the model/pipeline before full training'


## 4. VOC full training (120 epochs)

Resumes automatically from Drive if a previous session was interrupted.


In [ ]:
import os, shutil, subprocess, threading, time
os.makedirs(f'{RUNS}/voc', exist_ok=True)
if os.path.exists(f'{DRIVE}/runs/voc/last.pt') and not os.path.exists(f'{RUNS}/voc/last.pt'):
    shutil.copytree(f'{DRIVE}/runs/voc', f'{RUNS}/voc', dirs_exist_ok=True)
    print('restored VOC run state from Drive')

def _sync_loop():
    while True:
        try:
            shutil.copytree(f'{RUNS}/voc', f'{DRIVE}/runs/voc', dirs_exist_ok=True)
        except Exception:
            pass
        time.sleep(300)
threading.Thread(target=_sync_loop, daemon=True).start()

resume = f'--resume {RUNS}/voc/last.pt' if os.path.exists(f'{RUNS}/voc/last.pt') else ''
cmd = (f'python -m scripts.train --dataset voc --root {DATA}/VOC '
       f'--epochs 120 --batch-size 32 --img-size 640 --device cuda '
       f'--save-dir {RUNS}/voc {resume}')
print(cmd)
rc = subprocess.run(cmd, shell=True).returncode
print('rc =', rc)
assert rc == 0


In [ ]:
shutil.copytree(f'{RUNS}/voc', f'{DRIVE}/runs/voc', dirs_exist_ok=True)
print('VOC run synced to Drive')


## 5. VOC2007 test eval


In [ ]:
cmd = (f'python -m scripts.eval --dataset voc --root {DATA}/VOC '
       f'--weights {RUNS}/voc/best.pt --img-size 640 --device cuda')
print(cmd)
subprocess.run(cmd, shell=True)


## 6. COCO — data + full 300-epoch training (runs by default)

Runs the **full schedule**. A single Colab session cannot finish it (~1.5–3 days on T4):
on disconnect, re-run this notebook top-to-bottom — every stage resumes from Drive
(VOC is a no-op if complete; COCO continues from `last.pt`).


In [ ]:
from data.download import download_coco
download_coco(root=f'{DATA}/coco', splits=('val2017',), with_train=True)
for f in glob.glob(f'{DATA}/coco/*.zip'):
    os.remove(f)
print('COCO ready at', f'{DATA}/coco')


In [ ]:
os.makedirs(f'{RUNS}/coco', exist_ok=True)
if os.path.exists(f'{DRIVE}/runs/coco/last.pt') and not os.path.exists(f'{RUNS}/coco/last.pt'):
    shutil.copytree(f'{DRIVE}/runs/coco', f'{RUNS}/coco', dirs_exist_ok=True)
    print('restored COCO run state from Drive')

def _sync_loop_coco():
    while True:
        try:
            shutil.copytree(f'{RUNS}/coco', f'{DRIVE}/runs/coco', dirs_exist_ok=True)
        except Exception:
            pass
        time.sleep(300)
threading.Thread(target=_sync_loop_coco, daemon=True).start()

init = f'--init-from {RUNS}/voc/best.pt' if os.path.exists(f'{RUNS}/voc/best.pt') else ''
resume = f'--resume {RUNS}/coco/last.pt' if os.path.exists(f'{RUNS}/coco/last.pt') else ''
cmd = (f'python -m scripts.train --dataset coco --root {DATA}/coco '
       f'--epochs 300 --batch-size 64 --img-size 640 --device cuda '
       f'--save-dir {RUNS}/coco {init} {resume}')
print(cmd)
rc = subprocess.run(cmd, shell=True).returncode
print('rc =', rc, '(nonzero after a session timeout is expected — re-run to resume)')


In [ ]:
shutil.copytree(f'{RUNS}/coco', f'{DRIVE}/runs/coco', dirs_exist_ok=True)
print('COCO run synced to Drive')


## 7. COCO val2017 eval


In [ ]:
cmd = (f'python -m scripts.eval --dataset coco --root {DATA}/coco '
       f'--weights {RUNS}/coco/best.pt --img-size 640 --device cuda')
print(cmd)
subprocess.run(cmd, shell=True)


## 8. Export — ONNX (aux stripped) + INT8 + torch↔ORT parity


In [ ]:
BEST = f'{RUNS}/coco/best.pt' if os.path.exists(f'{RUNS}/coco/best.pt') else f'{RUNS}/voc/best.pt'
NC = 80 if 'coco' in BEST else 20
IMG = 640
print('exporting from', BEST, '| classes:', NC)
os.makedirs(f'{RUNS}/export', exist_ok=True)
cmd = (f'python -m export.onnx_export --weights {BEST} --out {RUNS}/export/evm_nano.onnx '
       f'--num-classes {NC} --img-size {IMG}')
print(cmd)
assert subprocess.run(cmd, shell=True).returncode == 0
from export.quantize import quantize_int8
quantize_int8(f'{RUNS}/export/evm_nano.onnx', f'{RUNS}/export/evm_nano_int8.onnx', img_size=IMG)
print('export done')


In [ ]:
import numpy as np
import torch
import onnxruntime as ort
from models import build_model
from export.decode_onnx import decode_outputs
from scripts.common import load_config
m = build_model(load_config('model_nano'), num_classes=NC)
sd = torch.load(BEST, map_location='cpu', weights_only=False)
m.load_state_dict(sd.get('model', sd), strict=True)
m.eval()
x = torch.randn(1, 3, IMG, IMG)
with torch.no_grad():
    res_t = m.predict(x, score_thresh=0.0, max_det=1000, use_obj=True)
sess = ort.InferenceSession(f'{RUNS}/export/evm_nano.onnx', providers=['CPUExecutionProvider'])
raw = sess.run(None, {'images': x.numpy()})
res_o = decode_outputs(raw, IMG, num_classes=NC, score_thresh=0.0, max_det=1000)
st = res_t[0]['scores'].numpy()
so = res_o[0]['scores']
print('max score torch / onnx:', float(st.max()), float(so.max()))
assert abs(float(st.max()) - float(so.max())) < 5e-3
print('PARITY OK')


## 9. CPU latency benchmarks — indicative only

Colab's CPU is **not** the deployment target. Final latency-vs-mAP table comes from
`Generic_Notebook.ipynb` (or the CLI) on the target laptop.


In [ ]:
cmd = (f'python -m benchmarks.bench_runtime --onnx {RUNS}/export/evm_nano.onnx '
       f'--img-size {IMG} --n-iter 50 --int8')
print(cmd)
subprocess.run(cmd, shell=True)


## 10. Curves + metrics.json + full Drive sync


In [ ]:
import json
import matplotlib.pyplot as plt
def curve(log_json, title):
    h = json.load(open(log_json))
    ep = [x['epoch'] for x in h]
    loss = [x['loss'] for x in h]
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].plot(ep, loss)
    ax[0].set_title(f'{title} loss')
    ax[0].set_xlabel('epoch')
    m = [(x['epoch'], x['mAP']) for x in h if 'mAP' in x]
    if m:
        ax[1].plot(*zip(*m))
        ax[1].set_title(f'{title} mAP')
        ax[1].set_xlabel('epoch')
    plt.tight_layout()
    plt.savefig(f'{RUNS}/{title.lower()}_curve.png', dpi=150)
    plt.show()
for ds in ('voc', 'coco'):
    try:
        curve(f'{RUNS}/{ds}/log.json', ds.upper())
    except FileNotFoundError:
        print(ds, 'log missing')


In [ ]:
metrics = {}
for ds, total_ep in (('voc', 120), ('coco', 300)):
    try:
        h = json.load(open(f'{RUNS}/{ds}/log.json'))
        maps = [x['mAP'] for x in h if 'mAP' in x]
        metrics[ds] = {'best_mAP': max(maps) if maps else None,
                      'epochs_done': h[-1]['epoch'] + 1,
                      'complete': h[-1]['epoch'] + 1 >= total_ep}
    except FileNotFoundError:
        metrics[ds] = None
for p in glob.glob(f'{RUNS}/export/*.onnx'):
    metrics[os.path.basename(p)] = os.path.getsize(p)
with open(f'{RUNS}/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=1)
print(json.dumps(metrics, indent=1))


In [ ]:
for d in ('overfit', 'voc', 'coco', 'export'):
    src = f'{RUNS}/{d}'
    if os.path.exists(src):
        shutil.copytree(src, f'{DRIVE}/runs/{d}', dirs_exist_ok=True)
shutil.copy(f'{RUNS}/metrics.json', f'{DRIVE}/metrics.json')
print('all artifacts synced to Drive:', DRIVE)
